# 6.2 TensorFlow / Keras → ONNX — Deep Dive

## Table of Contents
1. [The TF → ONNX Pipeline](#section-1)
2. [Graph Freezing](#section-2)
3. [Control Flow Conversion](#section-3)
4. [SavedModel Structure](#section-4)
5. [tf2onnx Conversion Paths](#section-5)
6. [Data Layout Conversion: NHWC ↔ NCHW](#section-6)
7. [Building and Converting a Keras Model](#section-7)
8. [Numerical Parity Checking](#section-8)
9. [Common Issues and Workarounds](#section-9)
10. [Key Takeaways](#section-10)

<a id='section-1'></a>
## Section 1: The TF → ONNX Pipeline

Converting TensorFlow models to ONNX follows a fundamentally different path
from PyTorch. TensorFlow models are already represented as **static computation
graphs** (even in TF2 with eager mode, the serving format is a frozen graph),
so the conversion is a **graph-to-graph translation** rather than a trace.

```
┌──────────────────────────────────────────────────────────────────────────┐
│                   TensorFlow → ONNX PIPELINE                            │
├──────────────────────────────────────────────────────────────────────────┤
│                                                                          │
│   Source Format         tf2onnx Processing           Output              │
│  ┌──────────────┐      ┌──────────────────┐      ┌───────────┐         │
│  │  SavedModel  │      │ 1. Load graph    │      │           │         │
│  │  ────────────│      │ 2. Freeze vars   │      │  .onnx    │         │
│  │  Keras .h5   │─────▶│ 3. Map TF ops    │─────▶│  file     │         │
│  │  ────────────│      │ 4. Layout convert│      │           │         │
│  │  Frozen .pb  │      │ 5. Optimize      │      │           │         │
│  └──────────────┘      └──────────────────┘      └───────────┘         │
│                                                                          │
│  Key tool: tf2onnx (Microsoft-supported, community-driven)              │
│  Reads: SavedModel (recommended), Frozen GraphDef, Keras H5, TFLite    │
└──────────────────────────────────────────────────────────────────────────┘
```

### Mathematical View

The conversion can be expressed as a graph homomorphism $\phi$:

$$\phi: G_{\text{TF}} \to G_{\text{ONNX}}$$

where for each TF operation $\text{op}_i \in G_{\text{TF}}$, there exists a
mapping:

$$\phi(\text{op}_i) = \{\text{onnx\_node}_1, \ldots, \text{onnx\_node}_k\}$$

The mapping preserves data flow edges — if $\text{op}_a \to \text{op}_b$ in
the TF graph, then the ONNX subgraph for $a$ feeds into the subgraph for $b$.

<a id='section-2'></a>
## Section 2: Graph Freezing

### What is Graph Freezing?

In TensorFlow, learned model parameters are stored as **Variables** — mutable
tensors that can be updated during training. At inference time, these values
are fixed. **Freezing** converts all Variables into Constants, producing a
self-contained, immutable graph.

```
┌─────────────────────────────────────────────────────────────────┐
│                  GRAPH FREEZING PROCESS                         │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  BEFORE (Training Graph):                                       │
│  ┌──────────────────────────────────────────┐                  │
│  │  Input ──▶ MatMul ──▶ Add ──▶ Relu ──▶ Output              │
│  │              ▲         ▲                                     │
│  │           Variable  Variable                                │
│  │           (weights) (bias)                                  │
│  │              ▲         ▲                                     │
│  │           Assign    Assign   ◄── Training ops               │
│  │              ▲         ▲                                     │
│  │           Optimizer  Optimizer                               │
│  └──────────────────────────────────────────┘                  │
│                                                                 │
│  AFTER (Frozen Graph):                                          │
│  ┌──────────────────────────────────────────┐                  │
│  │  Input ──▶ MatMul ──▶ Add ──▶ Relu ──▶ Output              │
│  │              ▲         ▲                                     │
│  │           Constant  Constant                                │
│  │           [0.3, ..] [0.1, ..]                               │
│  └──────────────────────────────────────────┘                  │
│                                                                 │
│  Removed: Assign ops, optimizer subgraph, gradient ops         │
│  Result:  Smaller, faster, no mutable state                    │
└─────────────────────────────────────────────────────────────────┘
```

### Formally

Given a training graph $G_{\text{train}} = (V, E)$ with variable nodes
$V_{\text{var}} \subset V$ and training-only nodes $V_{\text{train}} \subset V$
(optimizers, gradient ops, assign ops):

$$G_{\text{frozen}} = G_{\text{train}} \setminus V_{\text{train}}, \quad \text{with } V_{\text{var}} \to V_{\text{const}}$$

Each variable $v \in V_{\text{var}}$ with current value $\theta_v$ becomes a
constant node holding $\theta_v$ as an attribute.

### Why tf2onnx Needs Freezing

ONNX graphs are inherently frozen — they have `initializer` tensors (constants)
but no concept of mutable variables. The conversion must:

1. Evaluate all variable values from the checkpoint
2. Embed them as ONNX initializers
3. Strip all training infrastructure

<a id='section-3'></a>
## Section 3: Control Flow Conversion

TensorFlow and ONNX both support control flow, but with different primitives.
The converter must translate between them.

### TF While → ONNX Loop

TensorFlow's `tf.while_loop` becomes an ONNX `Loop` node. The mapping:

```
TF While Loop:                    ONNX Loop:
──────────────                   ──────────

tf.while_loop(                   Loop(
  cond=lambda i, x: i < N,        trip_count = N,
  body=lambda i, x: (i+1, f(x)),  cond = "",  (or condition subgraph)
  loop_vars=[0, x0]                body = subgraph { ... f(x) ... }
)                                )
```

The ONNX Loop operator takes:
- `trip_count`: maximum iterations (optional)
- `cond`: a boolean condition subgraph
- `body`: the loop body as a subgraph attribute

### TF If → ONNX If

```
TF Conditional:                   ONNX If:
───────────────                  ─────────

tf.cond(                         If(
  pred=condition,                  input: condition,
  true_fn=lambda: branch_a(x),     then_branch = subgraph { branch_a },
  false_fn=lambda: branch_b(x)     else_branch = subgraph { branch_b }
)                                )
```

### Switch/Merge Patterns (TF1)

In TF1 graphs, control flow uses low-level `Switch` and `Merge` primitives.
These are more complex to convert:

$$\text{Switch}(\text{data}, \text{pred}) \to \begin{cases} \text{output\_false} & \text{if pred is False} \\ \text{output\_true} & \text{if pred is True} \end{cases}$$

$$\text{Merge}(\text{inputs}) \to \text{first available input}$$

tf2onnx detects Switch/Merge patterns and lifts them into structured
`If` nodes. This pattern detection is one of the most complex parts
of the converter.

### Coverage Limitations

Not all TF control flow patterns convert cleanly:

| TF Pattern | ONNX Support | Notes |
|-----------|-------------|-------|
| Simple `tf.cond` | Good | Direct If mapping |
| `tf.while_loop` (static) | Good | Loop with trip count |
| Nested control flow | Partial | May require restructuring |
| `tf.case` (multi-branch) | Partial | Nested If nodes |
| Dynamic `tf.while_loop` | Partial | Condition subgraph mapping |

<a id='section-4'></a>
## Section 4: SavedModel Structure

The **SavedModel** is TensorFlow's recommended serialization format and the
preferred input for tf2onnx.

### Directory Layout

```
saved_model_dir/
├── saved_model.pb          # Graph definition + metadata
├── variables/
│   ├── variables.index     # Shard index for variable data
│   └── variables.data-00000-of-00001  # Variable values (checkpoint)
└── assets/                 # Optional: vocab files, lookup tables, etc.
    └── vocab.txt
```

### Signatures

A SavedModel can contain multiple **signatures** — named entry points
that define input/output tensor mappings:

| Signature | Purpose |
|-----------|--------|
| `serving_default` | Primary inference entry point |
| `train` | Training entry point (rarely exported) |
| Custom names | Multi-task models, feature extraction, etc. |

### Concrete Functions

Each signature maps to a **ConcreteFunction** — a frozen, traced TF graph
with known input/output shapes and types:

$$\text{Signature} \xrightarrow{\text{maps to}} \text{ConcreteFunction}(\text{inputs} \to \text{outputs})$$

When tf2onnx loads a SavedModel, it:
1. Looks for the `serving_default` signature (or user-specified one)
2. Extracts the corresponding ConcreteFunction
3. Freezes variables using checkpoint values
4. Converts the resulting graph to ONNX

### Key Considerations

- **Always export with `serving_default`** — this is what tf2onnx expects
- **Avoid resource ops** — Dataset iterators, hash tables, and other
  resource-based ops don't have ONNX equivalents
- **TF1 frozen `.pb` files** — still supported but require explicit
  `--inputs` and `--outputs` specification

<a id='section-5'></a>
## Section 5: tf2onnx Conversion Paths

tf2onnx provides two primary interfaces: a **CLI** for batch conversion
and a **Python API** for programmatic use.

### CLI Path (Recommended for SavedModel)

```bash
# From SavedModel directory
python -m tf2onnx.convert \
    --saved-model ./saved_model_dir \
    --output model.onnx \
    --opset 17

# From frozen graph (.pb)
python -m tf2onnx.convert \
    --input frozen_graph.pb \
    --inputs input:0 \
    --outputs output:0 \
    --output model.onnx \
    --opset 17

# With explicit signature
python -m tf2onnx.convert \
    --saved-model ./saved_model_dir \
    --signature_def serving_default \
    --output model.onnx \
    --opset 17
```

### Python API Path

```python
import tf2onnx
import tensorflow as tf

# From Keras model
spec = (tf.TensorSpec((None, 28, 28, 1), tf.float32, name="input"),)
model_proto, external_storage = tf2onnx.convert.from_keras(
    keras_model,
    input_signature=spec,
    opset=17,
    output_path="model.onnx"  # optional: save directly
)

# From ConcreteFunction
model_proto, _ = tf2onnx.convert.from_function(
    concrete_func,
    input_signature=spec,
    opset=17
)
```

### CLI Parameters Reference

| Parameter | Purpose |
|-----------|--------|
| `--saved-model` | Path to SavedModel directory |
| `--input` | Path to frozen `.pb` graph |
| `--output` | Output `.onnx` file path |
| `--opset` | Target ONNX opset version |
| `--inputs` / `--outputs` | Explicit I/O tensor names (for `.pb`) |
| `--signature_def` | Signature name (default: `serving_default`) |
| `--tag` | SavedModel tag (default: `serve`) |
| `--concrete_function` | Index of concrete function to convert |
| `--large_model` | Support models > 2 GB via external data |

<a id='section-6'></a>
## Section 6: Data Layout Conversion — NHWC ↔ NCHW

This is arguably the most important TF-specific concern in ONNX conversion.
TensorFlow and ONNX use different default data layouts for image tensors.

### The Layout Difference

```
NHWC (TensorFlow default):          NCHW (ONNX Conv default):
──────────────────────────          ─────────────────────────

Axis:  [Batch, Height, Width, Ch]   [Batch, Ch, Height, Width]
Index: [  0  ,   1   ,   2  , 3 ]   [  0  ,  1 ,   2  ,   3 ]

Example: 4D tensor for RGB 224×224 image
NHWC: [batch, 224, 224, 3]          NCHW: [batch, 3, 224, 224]
```

### The Permutation

The conversion between layouts is a tensor transpose (permutation):

$$X_{\text{NCHW}}[n, c, h, w] = X_{\text{NHWC}}[n, h, w, c]$$

Equivalently, using permutation notation:

$$X_{\text{NCHW}} = \text{Transpose}(X_{\text{NHWC}},\; \text{perm}=[0, 3, 1, 2])$$

$$X_{\text{NHWC}} = \text{Transpose}(X_{\text{NCHW}},\; \text{perm}=[0, 2, 3, 1])$$

### tf2onnx Transpose Insertion Strategy

tf2onnx handles this automatically by inserting `Transpose` nodes at
strategic points in the graph:

```
┌──────────────────────────────────────────────────────────────┐
│           TRANSPOSE INSERTION STRATEGY                       │
├──────────────────────────────────────────────────────────────┤
│                                                              │
│  TF Graph (NHWC):                                            │
│  Input[N,H,W,C] → Conv2D(NHWC) → MaxPool(NHWC) → Output    │
│                                                              │
│  ONNX Graph (NCHW):                                          │
│  Input[N,H,W,C]                                              │
│    → Transpose(0,3,1,2) → [N,C,H,W]                        │
│    → Conv(NCHW)                                              │
│    → MaxPool(NCHW)                                           │
│    → Transpose(0,2,3,1) → [N,H,W,C]                        │
│    → Output                                                  │
│                                                              │
│  Optimization: consecutive transposes are fused/cancelled    │
│  e.g., T(0,3,1,2) followed by T(0,2,3,1) = identity        │
└──────────────────────────────────────────────────────────────┘
```

### Performance Implications

The inserted transposes add overhead. For latency-sensitive deployments:

1. **Accept NCHW input** — modify your preprocessing to produce NCHW tensors,
   then tf2onnx can optimize away the input transpose
2. **Train in NCHW** — TF supports `data_format='channels_first'` for Conv2D,
   eliminating the need for layout conversion entirely
3. **Let the runtime optimize** — ONNX Runtime's graph optimizer can often
   fuse or eliminate redundant transposes

In [ ]:
import numpy as np

batch, height, width, channels = 2, 4, 4, 3
x_nhwc = np.arange(batch * height * width * channels, dtype=np.float32)
x_nhwc = x_nhwc.reshape(batch, height, width, channels)

x_nchw = np.transpose(x_nhwc, (0, 3, 1, 2))

print(f"NHWC shape: {x_nhwc.shape}  →  (batch, height, width, channels)")
print(f"NCHW shape: {x_nchw.shape}  →  (batch, channels, height, width)")

print(f"\nSample pixel at [0, 2, 3, :] in NHWC: {x_nhwc[0, 2, 3, :]}")
print(f"Same pixel at [0, :, 2, 3] in NCHW:    {x_nchw[0, :, 2, 3]}")
print(f"Values match: {np.array_equal(x_nhwc[0, 2, 3, :], x_nchw[0, :, 2, 3])}")

x_roundtrip = np.transpose(x_nchw, (0, 2, 3, 1))
print(f"\nRound-trip (NHWC→NCHW→NHWC) matches: {np.array_equal(x_nhwc, x_roundtrip)}")

<a id='section-7'></a>
## Section 7: Building and Converting a Keras Model

Let's walk through a complete example of building a Keras CNN and
converting it to ONNX.

### Architecture

```
Input: [batch, 28, 28, 1]  (NHWC — TF default)
  │
  ├─▶ Conv2D(16, 3×3, pad=same, relu)  →  [batch, 28, 28, 16]
  ├─▶ BatchNormalization                →  [batch, 28, 28, 16]
  ├─▶ MaxPooling2D(2×2)                 →  [batch, 14, 14, 16]
  │
  ├─▶ Conv2D(32, 3×3, pad=same, relu)  →  [batch, 14, 14, 32]
  ├─▶ BatchNormalization                →  [batch, 14, 14, 32]
  ├─▶ MaxPooling2D(2×2)                 →  [batch, 7, 7, 32]
  │
  ├─▶ Flatten                           →  [batch, 1568]
  ├─▶ Dense(128, relu)                  →  [batch, 128]
  ├─▶ Dropout(0.3)                      →  [batch, 128]  (identity in inference)
  └─▶ Dense(10)                         →  [batch, 10]
```

> **Note:** All TensorFlow code is wrapped in try/except blocks since
> TensorFlow may not be installed in your environment.

In [ ]:
try:
    import tensorflow as tf
    HAS_TF = True
    print(f"TensorFlow version: {tf.__version__}")
except ImportError:
    HAS_TF = False
    print("TensorFlow not installed.")
    print("Install with: pip install tensorflow")

try:
    import tf2onnx
    HAS_TF2ONNX = True
    print(f"tf2onnx version: {tf2onnx.__version__}")
except ImportError:
    HAS_TF2ONNX = False
    print("tf2onnx not installed.")
    print("Install with: pip install tf2onnx")

import numpy as np
import onnx
from onnx import checker

In [ ]:
if HAS_TF:
    inputs = tf.keras.Input(shape=(28, 28, 1), name="image")

    x = tf.keras.layers.Conv2D(16, 3, padding="same", activation="relu")(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D(pool_size=2)(x)

    x = tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D(pool_size=2)(x)

    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(10, name="logits")(x)

    keras_model = tf.keras.Model(inputs=inputs, outputs=outputs, name="mnist_cnn")
    keras_model.summary()
else:
    print("Skipping model creation — TensorFlow not available.")

In [ ]:
if HAS_TF and HAS_TF2ONNX:
    spec = (tf.TensorSpec((None, 28, 28, 1), tf.float32, name="image"),)

    model_proto, _ = tf2onnx.convert.from_keras(
        keras_model,
        input_signature=spec,
        opset=17,
    )

    checker.check_model(model_proto)

    num_nodes = len(model_proto.graph.node)
    unique_ops = sorted(set(n.op_type for n in model_proto.graph.node))

    print(f"Conversion successful!")
    print(f"  Nodes:      {num_nodes}")
    print(f"  Unique ops: {unique_ops}")
    print(f"  Opset:      {model_proto.opset_import[0].version}")

    transpose_count = sum(1 for n in model_proto.graph.node if n.op_type == "Transpose")
    print(f"  Transpose nodes (from NHWC→NCHW): {transpose_count}")

    print("\nGraph inputs:")
    for inp in model_proto.graph.input:
        dims = [d.dim_param or str(d.dim_value)
                for d in inp.type.tensor_type.shape.dim]
        print(f"  {inp.name}: [{', '.join(dims)}]")
    print("Graph outputs:")
    for out in model_proto.graph.output:
        dims = [d.dim_param or str(d.dim_value)
                for d in out.type.tensor_type.shape.dim]
        print(f"  {out.name}: [{', '.join(dims)}]")
else:
    print("Skipping conversion — TensorFlow or tf2onnx not available.")

<a id='section-8'></a>
## Section 8: Numerical Parity Checking

As with any framework conversion, verifying numerical equivalence between
the source TF model and the ONNX model is essential.

### Parity Testing Protocol for TF → ONNX

1. Generate random test inputs in **NHWC** format (TF native)
2. Run inference through the Keras model with `training=False`
3. Feed the **same tensor** to the ONNX model via ONNX Runtime
4. Compare outputs element-wise

### Tolerance Expectations

TF→ONNX parity tends to be slightly worse than PyTorch→ONNX because
of the layout transpositions and different BatchNorm implementations:

$$\text{Typical max diff: } \epsilon \approx 10^{-6} \text{ to } 10^{-5}$$

Differences above $10^{-4}$ warrant investigation.

In [ ]:
if HAS_TF and HAS_TF2ONNX:
    try:
        import onnxruntime as ort

        raw_bytes = model_proto.SerializeToString()
        sess = ort.InferenceSession(raw_bytes, providers=["CPUExecutionProvider"])

        ort_input_name = sess.get_inputs()[0].name
        ort_input_shape = sess.get_inputs()[0].shape
        print(f"ORT input: name='{ort_input_name}', shape={ort_input_shape}")

        print(f"\n{'Batch':>6} {'Max Abs Diff':>14} {'Mean Abs Diff':>14} {'Pass':>6}")
        print("-" * 44)

        for batch_size in [1, 4, 8, 16]:
            x_test = np.random.randn(batch_size, 28, 28, 1).astype(np.float32)

            y_tf = keras_model(x_test, training=False).numpy()

            y_ort = sess.run(None, {ort_input_name: x_test})[0]

            max_diff = np.abs(y_tf - y_ort).max()
            mean_diff = np.abs(y_tf - y_ort).mean()
            passed = max_diff < 1e-4

            print(f"{batch_size:>6} {max_diff:>14.2e} {mean_diff:>14.2e} {'OK' if passed else 'FAIL':>6}")

    except ImportError:
        print("onnxruntime not installed — install with: pip install onnxruntime")
else:
    print("Skipping parity check — TensorFlow or tf2onnx not available.")

### SavedModel Export Best Practices

When preparing a TF model for ONNX conversion, how you save the
model matters. Here are patterns that lead to clean conversions:

```python
# GOOD: Explicit input signature
@tf.function(input_signature=[
    tf.TensorSpec([None, 224, 224, 3], tf.float32, name='image')
])
def serve(self, image):
    return self.model(image, training=False)

tf.saved_model.save(model, './saved_model',
    signatures={'serving_default': model.serve})
```

### Common Anti-Patterns

| Anti-Pattern | Why It's Bad | Fix |
|-------------|-------------|-----|
| No input signature | Unknown input shapes/types | Add `input_signature` to `@tf.function` |
| Training ops in serving graph | Conversion includes optimizer | Use `training=False` |
| Python-only preprocessing | Can't be in ONNX graph | Use `tf.keras.layers` for preprocessing |
| Multiple signatures without specifying which | Wrong graph converted | Use `--signature_def` |

In [ ]:
if HAS_TF:
    print("SavedModel inspection example:")
    print()

    import tempfile, os

    with tempfile.TemporaryDirectory() as tmp:
        sm_path = os.path.join(tmp, "saved_model")
        tf.saved_model.save(keras_model, sm_path)

        loaded = tf.saved_model.load(sm_path)
        signatures = list(loaded.signatures.keys())
        print(f"  Signatures: {signatures}")

        if 'serving_default' in loaded.signatures:
            sig = loaded.signatures['serving_default']
            print(f"  Inputs:")
            for name, spec in sig.structured_input_signature[1].items():
                print(f"    {name}: shape={spec.shape}, dtype={spec.dtype.name}")
            print(f"  Outputs:")
            for name, tensor in sig.structured_outputs.items():
                print(f"    {name}: shape={tensor.shape}, dtype={tensor.dtype.name}")

        contents = []
        for root, dirs, files in os.walk(sm_path):
            for f in files:
                full = os.path.join(root, f)
                rel = os.path.relpath(full, sm_path)
                size = os.path.getsize(full)
                contents.append((rel, size))

        print(f"\n  SavedModel contents:")
        for rel, size in sorted(contents):
            print(f"    {rel}: {size:,} bytes")
else:
    print("Skipping — TensorFlow not available.")

### Graph Optimization in tf2onnx

tf2onnx applies several graph optimizations during conversion to produce
cleaner, more efficient ONNX graphs:

1. **Identity elimination** — removes pass-through Identity nodes
2. **Transpose cancellation** — fuses consecutive inverse transposes
3. **Constant folding** — pre-computes static subexpressions
4. **Node merging** — combines compatible adjacent nodes

These optimizations are automatic but can be controlled:

```bash
# Skip optimizations (for debugging)
python -m tf2onnx.convert --saved-model ./model \
    --output model.onnx --opset 17 \
    --optimization_level none
```

### Transpose Optimization Impact

For CNNs, the transpose optimizer is critical. Without it, a 4-layer
CNN might have 8+ Transpose nodes. After optimization, many cancel out:

$$N_{\text{transpose}}^{\text{optimized}} \leq 2 \quad \text{(input + output only)}$$

In [ ]:
if HAS_TF and HAS_TF2ONNX:
    print("Analyzing ONNX graph structure from TF conversion:")
    print()

    op_counts = {}
    for node in model_proto.graph.node:
        op_counts[node.op_type] = op_counts.get(node.op_type, 0) + 1

    print(f"  {'Operator':<25} {'Count':>6}")
    print(f"  {'-' * 25} {'-' * 6}")
    for op, count in sorted(op_counts.items(), key=lambda x: -x[1]):
        print(f"  {op:<25} {count:>6}")

    print(f"\n  Total nodes: {sum(op_counts.values())}")
    print(f"  Transpose nodes: {op_counts.get('Transpose', 0)}")
    print(f"  Initializers: {len(model_proto.graph.initializer)}")
else:
    print("Skipping — TensorFlow or tf2onnx not available.")

<a id='section-9'></a>
## Section 9: Common Issues and Workarounds

### Issue Reference Table

| # | Issue | Symptom | Root Cause | Fix |
|---|-------|---------|------------|-----|
| 1 | Unknown TF op | `ValueError: Cannot convert op` | Custom or very new kernel without ONNX mapping | Upgrade tf2onnx; replace op with supported equivalent |
| 2 | Shape `?` everywhere | All dims show as unknown | Dynamic shapes not properly constrained | Freeze known dims; use `--inputs` with shapes |
| 3 | Transpose storm | Many extra Transpose nodes | NHWC→NCHW layout conversion | Accept (runtime optimizes) or train in NCHW |
| 4 | Signature mismatch | Wrong inputs/outputs selected | Multiple signatures in SavedModel | Pass `--signature_def serving_default` explicitly |
| 5 | Resource ops | `ResourceGather not supported` | Lookup tables, iterators in graph | Export inference-only; remove preprocessing from model |
| 6 | Large model (> 2 GB) | Protobuf serialization fails | ONNX single-file limit is 2 GB | Use `--large_model` flag for external data format |
| 7 | Version mismatch | Unexpected conversion errors | TF version incompatible with tf2onnx | Check tf2onnx compatibility matrix |
| 8 | Training ops in graph | Conversion fails or wrong results | Model not in inference mode | Ensure `training=False` in Keras export |

### Debugging Conversion Failures

When tf2onnx fails, a systematic debugging approach:

```
┌─────────────────────────────────────────────────────────────┐
│            DEBUGGING CONVERSION FAILURES                    │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  1. Add --verbose flag to CLI conversion                    │
│     → Shows which ops are being converted/skipped           │
│                                                             │
│  2. Check tf2onnx supported ops list                        │
│     → github.com/onnx/tensorflow-onnx/blob/main/support     │
│                                                             │
│  3. Isolate the failing op                                  │
│     → Build minimal model with just that op                 │
│     → Test conversion alone                                 │
│                                                             │
│  4. Try a different opset version                            │
│     → Some ops gain support in higher opsets                 │
│                                                             │
│  5. Replace the unsupported op                               │
│     → Use equivalent supported operations                   │
│     → Register custom op handler in tf2onnx                 │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
if HAS_TF:
    print("TF op categories relevant to ONNX conversion:")
    print()

    categories = {
        "Well-supported": [
            "Conv2D", "Dense (MatMul+Add)", "MaxPool", "AvgPool",
            "BatchNormalization", "Relu/Sigmoid/Tanh", "Softmax",
            "Reshape", "Flatten", "Concat", "Add/Mul/Sub",
        ],
        "Partial support": [
            "LSTM/GRU (static unroll)", "tf.cond", "tf.while_loop",
            "SparseTensor ops", "RaggedTensor ops",
        ],
        "Typically unsupported": [
            "tf.data.Dataset ops", "tf.lookup (hash tables)",
            "tf.io (file I/O)", "Custom C++ ops",
            "tf.distribute (multi-GPU)",
        ],
    }

    for category, ops in categories.items():
        print(f"  {category}:")
        for op in ops:
            print(f"    - {op}")
        print()
else:
    print("TF not available — see table above for op support categories.")

<a id='section-10'></a>
## Section 10: Key Takeaways

### Conversion Checklist

```
┌─────────────────────────────────────────────────────────────────┐
│           TensorFlow → ONNX CONVERSION CHECKLIST               │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  □  1. Save model as SavedModel format (preferred input)        │
│  □  2. Verify serving_default signature exports correctly       │
│  □  3. Remove training-only ops (Dataset, optimizer) from graph │
│  □  4. Choose target opset matching your runtime                │
│  □  5. Run tf2onnx conversion (CLI or Python API)               │
│  □  6. Validate with onnx.checker.check_model()                 │
│  □  7. Check for excessive Transpose nodes                      │
│  □  8. Verify numerical parity (max diff < 1e-4 for float32)   │
│  □  9. Test with multiple input shapes if dynamic               │
│  □ 10. Include parity test in CI pipeline                       │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### Core Concepts Summary

| Concept | Key Insight |
|---------|------------|
| **Graph Freezing** | Converts TF Variables to constants; removes training subgraph; required for ONNX |
| **Layout Conversion** | TF uses NHWC; ONNX Conv uses NCHW; tf2onnx inserts Transpose nodes automatically |
| **Control Flow** | TF `tf.cond`→ONNX `If`, TF `tf.while_loop`→ONNX `Loop`; coverage is good but not universal |
| **SavedModel** | Preferred source format; contains signatures mapping to ConcreteFunction graphs |
| **tf2onnx** | Primary converter tool; CLI and Python API; supports SavedModel, frozen .pb, Keras H5 |
| **Parity** | Expect $\max|\Delta| < 10^{-5}$; layout transpositions may introduce slightly larger diffs |